# 99 — Build hierarchically integrated virtual T1 shot gathers (SAFE, v3)

Each virtual shot is assembled sequentially:

1. T1_1m establishes the preferred base gather.
2. T1_2m is correlated, aligned and normally super-stacked at coincident receivers.
3. Nodal data is correlated and aligned; by default it only fills unoccupied receivers.
4. T1_Streamer is correlated and aligned; by default it only fills unoccupied receivers.

Alignment is performed before output trimming and can search up to +/-0.5 s.
The common output duration is independent of the correlation window and is
configurable below.

## 1. Configuration

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import math

import numpy as np
import pandas as pd
from scipy.signal import hilbert

from obspy import read, Stream, Trace, UTCDateTime

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
PLAN_ROOT = PROJECT_ROOT / '98_virtual_T1_integration_plan'
OUT_ROOT = PROJECT_ROOT / '99_virtual_T1_shot_gathers'
MSEED_ROOT = OUT_ROOT / 'per_shot_mseed'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
MSEED_ROOT.mkdir(parents=True, exist_ok=True)

SHOT_CATALOG_PATH = PLAN_ROOT / '98_virtual_T1_shot_catalog.csv'
PRODUCT_PLAN_PATH = PLAN_ROOT / '98_virtual_T1_product_plan.csv'

COMPONENT = 'Z'
TARGET_SAMPLING_RATE_HZ = 1000.0

# Common final output interval. Longer secondary records are aligned in full
# before this interval is extracted. A T1_1m trace may contain NaNs/zeros after
# its approximately 0.4 s record ends; longer products retain later energy.
OUTPUT_START_S = -0.025
OUTPUT_END_S = 0.600

RECEIVER_MATCH_TOLERANCE_M = 0.25

# Requested hierarchy and default super-stack policy.
INTEGRATION_PRIORITY = {
    'T1_1m': 10,
    'T1_2m': 20,
    'nodal': 30,
    'T1_Streamer': 40,
}
T1_2m_SUPER_STACK = True
T1_NODAL_SUPER_STACK = False
T1_STREAMER_SUPER_STACK = False
SUPER_STACK_BY_ROLE = {
    'T1_1m': False,
    'T1_2m': T1_2m_SUPER_STACK,
    'nodal': T1_NODAL_SUPER_STACK,
    'T1_Streamer': T1_STREAMER_SUPER_STACK,
}

# Alignment is two-stage. Coarse alignment uses smoothed envelopes and can move
# a shorter 0.4 s reference within records as long as the configured search
# allows. Fine alignment uses the signed waveform close to the coarse solution.
MAX_COARSE_ALIGNMENT_SHIFT_S = 0.500
COARSE_ALIGNMENT_STEP_S = 0.005
MAX_FINE_ALIGNMENT_SHIFT_S = 0.020
FINE_ALIGNMENT_STEP_S = 0.001
ENVELOPE_SMOOTH_S = 0.008
MIN_ALIGNMENT_OVERLAP_S = 0.080
MIN_COMMON_RECEIVERS_FOR_ALIGNMENT = 2
ALIGNMENT_MIN_ENVELOPE_CORRELATION = 0.55
ALIGNMENT_MIN_WAVEFORM_CORRELATION = 0.35
SUPER_STACK_MIN_CORRELATION = 0.90

# Products that cannot be aligned are not silently incorporated. Set this True
# only for an intentional diagnostic build using their initial plan shift.
ALLOW_UNALIGNED_NEW_RECEIVERS = False

# Nodal stack weight reflects accepted repeated shots. Original individual
# Geode/streamer gathers have unit weight.
NODAL_WEIGHT_COLUMN = 'n_accepted_members'
RAW_GEODE_WEIGHT = 1.0

VIRTUAL_EPOCH = UTCDateTime(2000, 1, 1)
SHOT_TIME_STRIDE_S = 10.0
WRITE_ALL_SHOTS_MSEED = True
MSEED_ENCODING = 'FLOAT32'
REQUIRE_GEODE_OUTPUT_IF_PLANNED = True

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Output:', OUT_ROOT)
print('Integration priority:', INTEGRATION_PRIORITY)
print('Super-stack policy:', SUPER_STACK_BY_ROLE)
print('Alignment search: +/-', MAX_COARSE_ALIGNMENT_SHIFT_S, 's')
print('Output relative window:', OUTPUT_START_S, 'to', OUTPUT_END_S, 's')

Output: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers
Integration priority: {'T1_1m': 10, 'T1_2m': 20, 'nodal': 30, 'T1_Streamer': 40}
Super-stack policy: {'T1_1m': False, 'T1_2m': True, 'nodal': False, 'T1_Streamer': False}
Alignment search: +/- 0.5 s
Output relative window: -0.025 to 0.6 s


## 2. Load integration plan

In [2]:
for path in [SHOT_CATALOG_PATH, PRODUCT_PLAN_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing notebook-98 output: {path}')

shots = pd.read_csv(SHOT_CATALOG_PATH, low_memory=False)
products = pd.read_csv(PRODUCT_PLAN_PATH, low_memory=False)

products['include_in_virtual_shot'] = products.include_in_virtual_shot.astype(str).str.lower().isin(['true', '1', 'yes'])
products['initial_time_shift_s'] = pd.to_numeric(
    products.get('initial_time_shift_s', products.get('time_shift_to_canonical_s', 0.0)),
    errors='coerce',
).fillna(0.0)
products['source_x_m'] = pd.to_numeric(products.source_x_m, errors='coerce')
products['integration_role'] = products.get('integration_role', 'nodal').fillna('nodal')
products['integration_priority'] = pd.to_numeric(
    products.get('integration_priority', products.integration_role.map(INTEGRATION_PRIORITY)),
    errors='coerce',
).fillna(products.integration_role.map(INTEGRATION_PRIORITY)).astype(int)

products = products.loc[
    products.include_in_virtual_shot
    & products.component.astype(str).str.upper().eq(COMPONENT)
].copy()

unknown_roles = sorted(set(products.integration_role) - set(INTEGRATION_PRIORITY))
if unknown_roles:
    raise ValueError(f'Unknown integration roles: {unknown_roles}')

print('Virtual shots:', len(shots))
print('Included products:', len(products))
display(products.groupby(
    ['integration_role', 'product_kind', 'survey'], dropna=False
).size().reset_index(name='n_products'))

Virtual shots: 159
Included products: 354


,integration_role,product_kind,survey,n_products
0,T1_1m,geode_raw,T1_1m_refraction,39
1,T1_2m,geode_raw,T1_2m_refraction,36
2,T1_Streamer,geode_raw,T1_streamer_masw,80
3,nodal,nodal_stack,T1_1m_refraction,39
4,nodal,nodal_stack,T1_2m_refraction,36
5,nodal,nodal_stack,T1_streamer_masw,80
6,nodal,nodal_stack,NaN,44


## 3. Waveform and geometry helpers

In [3]:
def normalize_component(value):
    text = str(value).strip().upper()
    return text[-1] if text and text[-1] in 'ZNE' else text


def as_bool(value):
    return str(value).strip().lower() in {'true', '1', 'yes'}


def trace_receiver_x_m(trace, *, raw_geode=False):
    candidates = [
        getattr(trace.stats, 'receiver_x_m', np.nan),
        getattr(trace.stats, 'distance', np.nan),
    ]
    seg2 = getattr(trace.stats, 'seg2', None)
    if seg2 is not None:
        keys = ['RECEIVER_LOCATION'] if raw_geode else [
            'RECEIVER_LOCATION', 'RECEIVER_STATION_NUMBER'
        ]
        for key in keys:
            try:
                candidates.append(seg2.get(key, np.nan))
            except Exception:
                pass
    for candidate in candidates:
        value = pd.to_numeric(candidate, errors='coerce')
        if pd.notna(value):
            return float(value)
    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def read_product_stream(product):
    path = Path(str(product.waveform_path))
    if not path.exists():
        raise FileNotFoundError(path)
    stream = read(str(path))
    selected = Stream(
        tr.copy() for tr in stream
        if product.product_kind == 'geode_raw'
        or normalize_component(tr.stats.channel) == COMPONENT
    )
    if product.product_kind == 'geode_raw':
        for tr in selected:
            tr.stats.channel = 'GHZ'
    return selected


def assign_receiver_positions(stream, product):
    output = []
    fallback_first = pd.to_numeric(getattr(product, 'receiver_first_x_m_fallback', np.nan), errors='coerce')
    fallback_dx = pd.to_numeric(getattr(product, 'receiver_dx_m_fallback', np.nan), errors='coerce')
    reverse = as_bool(getattr(product, 'reverse_trace_order_fallback', False))
    indices = list(range(len(stream)))
    if reverse:
        indices.reverse()
    for output_index, original_index in enumerate(indices):
        trace = stream[original_index].copy()
        receiver_x = trace_receiver_x_m(trace, raw_geode=(product.product_kind == 'geode_raw'))
        if product.product_kind == 'geode_raw' and (
            not np.isfinite(receiver_x) or receiver_x < -1000 or receiver_x > 10000
        ):
            if pd.isna(fallback_first) or pd.isna(fallback_dx):
                continue
            receiver_x = float(fallback_first + output_index * fallback_dx)
        if np.isfinite(receiver_x):
            output.append((float(receiver_x), trace, original_index))
    return output


def product_weight(product):
    if product.product_kind == 'nodal_stack':
        value = pd.to_numeric(getattr(product, NODAL_WEIGHT_COLUMN, np.nan), errors='coerce')
        return float(value) if pd.notna(value) and value > 0 else 1.0
    return RAW_GEODE_WEIGHT


def prepare_trace(trace):
    working = trace.copy()
    working.detrend('demean')
    target_rate = float(TARGET_SAMPLING_RATE_HZ)
    if not np.isclose(working.stats.sampling_rate, target_rate):
        working.interpolate(sampling_rate=target_rate, method='lanczos', a=12)
    data = np.asarray(working.data, dtype=float)
    times = np.arange(data.size, dtype=float) / target_rate
    return times, data


def sample_shifted(input_times, input_data, output_times, shift_s):
    # Positive shift moves the waveform later: output(t)=input(t-shift).
    return np.interp(
        output_times - float(shift_s), input_times, input_data,
        left=np.nan, right=np.nan,
    )


def smooth_envelope(data):
    finite = np.isfinite(data)
    filled = np.where(finite, data, 0.0)
    envelope = np.abs(hilbert(filled))
    n = max(1, int(round(ENVELOPE_SMOOTH_S * TARGET_SAMPLING_RATE_HZ)))
    if n > 1:
        envelope = np.convolve(envelope, np.ones(n) / n, mode='same')
    envelope[~finite] = np.nan
    return envelope


def normalized_correlation(a, b):
    valid = np.isfinite(a) & np.isfinite(b)
    minimum = max(3, int(round(MIN_ALIGNMENT_OVERLAP_S * TARGET_SAMPLING_RATE_HZ)))
    if valid.sum() < minimum:
        return np.nan
    x = np.asarray(a[valid], dtype=float)
    y = np.asarray(b[valid], dtype=float)
    x -= x.mean(); y -= y.mean()
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denom) if denom > 0 else np.nan


def receiver_cluster_key(receiver_x_m):
    return int(round(float(receiver_x_m) / RECEIVER_MATCH_TOLERANCE_M))


def best_shift_for_pair(reference_data, candidate_times, candidate_data, initial_shift_s):
    reference_times = OUTPUT_TIMES
    reference_env = smooth_envelope(reference_data)
    candidate_env = smooth_envelope(candidate_data)

    coarse_offsets = np.arange(
        -MAX_COARSE_ALIGNMENT_SHIFT_S,
        MAX_COARSE_ALIGNMENT_SHIFT_S + COARSE_ALIGNMENT_STEP_S / 2,
        COARSE_ALIGNMENT_STEP_S,
    )
    coarse_scores = []
    for offset in coarse_offsets:
        shift = float(initial_shift_s + offset)
        sampled = sample_shifted(candidate_times, candidate_env, reference_times, shift)
        coarse_scores.append(normalized_correlation(reference_env, sampled))
    coarse_scores = np.asarray(coarse_scores, dtype=float)
    if not np.isfinite(coarse_scores).any():
        return None
    coarse_index = int(np.nanargmax(coarse_scores))
    coarse_shift = float(initial_shift_s + coarse_offsets[coarse_index])
    coarse_corr = float(coarse_scores[coarse_index])

    fine_offsets = np.arange(
        -MAX_FINE_ALIGNMENT_SHIFT_S,
        MAX_FINE_ALIGNMENT_SHIFT_S + FINE_ALIGNMENT_STEP_S / 2,
        FINE_ALIGNMENT_STEP_S,
    )
    fine_scores = []
    for offset in fine_offsets:
        shift = coarse_shift + float(offset)
        sampled = sample_shifted(candidate_times, candidate_data, reference_times, shift)
        fine_scores.append(normalized_correlation(reference_data, sampled))
    fine_scores = np.asarray(fine_scores, dtype=float)
    if np.isfinite(fine_scores).any():
        fine_index = int(np.nanargmax(fine_scores))
        fine_shift = float(coarse_shift + fine_offsets[fine_index])
        fine_corr = float(fine_scores[fine_index])
    else:
        fine_shift = coarse_shift
        fine_corr = np.nan
    return {
        'coarse_shift_s': coarse_shift,
        'coarse_envelope_corr': coarse_corr,
        'fine_shift_s': fine_shift,
        'fine_waveform_corr': fine_corr,
    }


def estimate_product_alignment(gather, candidate_traces, initial_shift_s):
    pair_results = []
    by_key = {receiver_cluster_key(item['receiver_x_m']): item for item in candidate_traces}
    for key, existing in gather.items():
        candidate = by_key.get(key)
        if candidate is None:
            continue
        result = best_shift_for_pair(
            existing['data'], candidate['input_times'], candidate['input_data'], initial_shift_s
        )
        if result is not None:
            result.update({
                'receiver_cluster_key': key,
                'existing_receiver_x_m': existing['receiver_x_m'],
                'candidate_receiver_x_m': candidate['receiver_x_m'],
            })
            pair_results.append(result)

    if not pair_results:
        return {
            'accepted': False, 'status': 'no_common_receivers',
            'n_common_receivers': 0, 'applied_shift_s': np.nan,
            'median_envelope_corr': np.nan, 'median_waveform_corr': np.nan,
            'pair_results': [],
        }

    frame = pd.DataFrame(pair_results)
    applied_shift = float(np.nanmedian(frame.fine_shift_s))
    median_env = float(np.nanmedian(frame.coarse_envelope_corr))
    median_wave = float(np.nanmedian(frame.fine_waveform_corr))
    enough = len(frame) >= MIN_COMMON_RECEIVERS_FOR_ALIGNMENT
    accepted = (
        enough
        and median_env >= ALIGNMENT_MIN_ENVELOPE_CORRELATION
        and (np.isnan(median_wave) or median_wave >= ALIGNMENT_MIN_WAVEFORM_CORRELATION)
    )
    return {
        'accepted': bool(accepted),
        'status': 'accepted' if accepted else ('insufficient_common_receivers' if not enough else 'low_correlation'),
        'n_common_receivers': len(frame),
        'applied_shift_s': applied_shift,
        'median_envelope_corr': median_env,
        'median_waveform_corr': median_wave,
        'pair_results': pair_results,
    }


def valid_weighted_stack(existing_data, candidate_data, existing_weight, candidate_weight):
    arrays = np.vstack([existing_data, candidate_data])
    weights = np.asarray([existing_weight, candidate_weight], dtype=float)
    finite = np.isfinite(arrays)
    numerator = np.where(finite, arrays * weights[:, None], 0.0).sum(axis=0)
    denominator = np.where(finite, weights[:, None], 0.0).sum(axis=0)
    return np.divide(numerator, denominator, out=np.full(arrays.shape[1], np.nan), where=denominator > 0)


def short_station_code(receiver_x_m):
    return f'R{int(round(receiver_x_m * 10)):04d}'[:5]


OUTPUT_TIMES = np.arange(
    OUTPUT_START_S,
    OUTPUT_END_S,
    1.0 / TARGET_SAMPLING_RATE_HZ,
)

## 4. Build every virtual shot

In [4]:
shot_rows = []
trace_rows = []
contribution_rows = []
alignment_rows = []
alignment_pair_rows = []
product_error_rows = []
all_shot_stream = Stream()

for shot in shots.sort_values('virtual_shot_number').itertuples(index=False):
    shot_products = products.loc[
        products.virtual_shot_number.eq(shot.virtual_shot_number)
    ].sort_values(['integration_priority', 'product_id'], kind='stable')

    gather = {}  # one authoritative output trace per receiver-position cluster
    product_errors = []
    n_loaded_candidate_traces = 0

    for product in shot_products.itertuples(index=False):
        try:
            stream = read_product_stream(product)
            positioned = assign_receiver_positions(stream, product)
            weight = product_weight(product)
            candidate_traces = []
            for receiver_x, trace, original_trace_index in positioned:
                input_times, input_data = prepare_trace(trace)
                candidate_traces.append({
                    'receiver_x_m': receiver_x,
                    'cluster_key': receiver_cluster_key(receiver_x),
                    'input_times': input_times,
                    'input_data': input_data,
                    'original_trace_index': original_trace_index,
                })
            n_loaded_candidate_traces += len(candidate_traces)

            role = str(product.integration_role)
            initial_shift = float(product.initial_time_shift_s)
            super_stack_enabled = bool(SUPER_STACK_BY_ROLE[role])

            if not gather:
                alignment = {
                    'accepted': True,
                    'status': 'base_product',
                    'n_common_receivers': 0,
                    'applied_shift_s': initial_shift,
                    'median_envelope_corr': np.nan,
                    'median_waveform_corr': np.nan,
                    'pair_results': [],
                }
            else:
                alignment = estimate_product_alignment(gather, candidate_traces, initial_shift)

            alignment_rows.append({
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'product_id': product.product_id,
                'integration_role': role,
                'integration_priority': int(product.integration_priority),
                'super_stack_enabled': super_stack_enabled,
                'initial_shift_s': initial_shift,
                'alignment_status': alignment['status'],
                'alignment_accepted': alignment['accepted'],
                'n_common_receivers': alignment['n_common_receivers'],
                'applied_shift_s': alignment['applied_shift_s'],
                'median_envelope_corr': alignment['median_envelope_corr'],
                'median_waveform_corr': alignment['median_waveform_corr'],
            })
            for pair in alignment['pair_results']:
                alignment_pair_rows.append({
                    'virtual_shot_number': int(shot.virtual_shot_number),
                    'product_id': product.product_id,
                    'integration_role': role,
                    **pair,
                })

            if not alignment['accepted'] and not ALLOW_UNALIGNED_NEW_RECEIVERS:
                continue
            applied_shift = (
                float(alignment['applied_shift_s'])
                if np.isfinite(alignment['applied_shift_s'])
                else initial_shift
            )

            pair_corr_by_key = {
                int(pair['receiver_cluster_key']): pair['fine_waveform_corr']
                for pair in alignment['pair_results']
            }

            for candidate in candidate_traces:
                key = candidate['cluster_key']
                shifted = sample_shifted(
                    candidate['input_times'], candidate['input_data'], OUTPUT_TIMES, applied_shift
                )
                contribution = {
                    'product_id': product.product_id,
                    'product_kind': product.product_kind,
                    'survey': product.survey,
                    'integration_role': role,
                    'waveform_path': product.waveform_path,
                    'input_receiver_x_m': candidate['receiver_x_m'],
                    'original_trace_index': candidate['original_trace_index'],
                    'applied_shift_s': applied_shift,
                    'weight': weight,
                }

                if key not in gather:
                    gather[key] = {
                        'receiver_x_m': candidate['receiver_x_m'],
                        'data': shifted,
                        'weight': weight,
                        'primary_role': role,
                        'primary_family': product.receiver_family,
                        'contributions': [contribution],
                    }
                    continue

                # Occupied location: lower-priority trace is ignored unless its
                # role permits super-stacking and its receiver-level waveform
                # correlation is exceptionally good.
                receiver_corr = pair_corr_by_key.get(key, np.nan)
                if super_stack_enabled and np.isfinite(receiver_corr) and receiver_corr >= SUPER_STACK_MIN_CORRELATION:
                    existing = gather[key]
                    existing['data'] = valid_weighted_stack(
                        existing['data'], shifted, existing['weight'], weight
                    )
                    existing['weight'] += weight
                    contribution['super_stacked'] = True
                    contribution['receiver_waveform_corr'] = receiver_corr
                    existing['contributions'].append(contribution)
                else:
                    # Explicitly record why a coincident lower-priority trace was not used.
                    contribution_rows.append({
                        'virtual_shot_number': int(shot.virtual_shot_number),
                        'virtual_shot_id': shot.virtual_shot_id,
                        'source_x_m': float(shot.source_x_m),
                        'output_receiver_x_m': gather[key]['receiver_x_m'],
                        **contribution,
                        'used_in_output': False,
                        'super_stacked': False,
                        'receiver_waveform_corr': receiver_corr,
                        'decision': 'occupied_super_stack_disabled_or_below_threshold',
                    })

        except Exception as exc:
            error_row = {
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'product_id': product.product_id,
                'product_kind': product.product_kind,
                'integration_role': getattr(product, 'integration_role', None),
                'receiver_family': product.receiver_family,
                'survey': product.survey,
                'waveform_path': product.waveform_path,
                'error': repr(exc),
            }
            product_errors.append(error_row)
            product_error_rows.append(error_row)

    output_stream = Stream()
    for trace_number, (key, entry) in enumerate(sorted(gather.items(), key=lambda item: item[1]['receiver_x_m']), start=1):
        receiver_x = float(entry['receiver_x_m'])
        data = np.nan_to_num(entry['data'], nan=0.0).astype(np.float32)
        trace = Trace(data=data)
        trace.stats.network = 'VT'
        trace.stats.station = short_station_code(receiver_x)
        role_code = {'T1_1m':'1M', 'T1_2m':'2M', 'nodal':'ND', 'T1_Streamer':'ST'}[entry['primary_role']]
        trace.stats.location = role_code
        trace.stats.channel = 'GHZ'
        trace.stats.sampling_rate = TARGET_SAMPLING_RATE_HZ
        trace.stats.starttime = VIRTUAL_EPOCH + shot.virtual_shot_number * SHOT_TIME_STRIDE_S + OUTPUT_START_S
        trace.stats.receiver_x_m = receiver_x
        trace.stats.source_x_m = float(shot.source_x_m)
        trace.stats.virtual_shot_number = int(shot.virtual_shot_number)
        trace.stats.receiver_family = entry['primary_family']
        trace.stats.integration_role = entry['primary_role']
        output_stream += trace

        used_ids=[]
        used_roles=[]
        for contribution in entry['contributions']:
            used_ids.append(str(contribution['product_id']))
            used_roles.append(str(contribution['integration_role']))
            contribution_rows.append({
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'output_receiver_x_m': receiver_x,
                **contribution,
                'used_in_output': True,
                'super_stacked': bool(contribution.get('super_stacked', False)),
                'receiver_waveform_corr': contribution.get('receiver_waveform_corr', np.nan),
                'decision': 'base_or_new_receiver' if not contribution.get('super_stacked', False) else 'super_stacked',
            })

        trace_rows.append({
            'virtual_shot_number': int(shot.virtual_shot_number),
            'virtual_shot_id': shot.virtual_shot_id,
            'source_cluster_id': shot.source_cluster_id,
            'source_x_m': float(shot.source_x_m),
            'trace_number_within_shot': trace_number,
            'receiver_family': entry['primary_family'],
            'integration_role': entry['primary_role'],
            'receiver_x_m': receiver_x,
            'station': trace.stats.station,
            'channel': trace.stats.channel,
            'sampling_rate_hz': TARGET_SAMPLING_RATE_HZ,
            'relative_start_s': OUTPUT_START_S,
            'relative_end_s': OUTPUT_END_S,
            'n_samples': trace.stats.npts,
            'n_contributing_products': len(entry['contributions']),
            'total_weight': float(entry['weight']),
            'contributing_product_ids': ' | '.join(used_ids),
            'contributing_roles': ' | '.join(used_roles),
        })

    shot_token = f'{int(shot.virtual_shot_number):04d}_x{float(shot.source_x_m):07.1f}m'
    mseed_path = MSEED_ROOT / f'T1_VIRTUAL_SHOT_{shot_token}_{COMPONENT}.mseed'
    if len(output_stream):
        output_stream.write(str(mseed_path), format='MSEED', encoding=MSEED_ENCODING)
        all_shot_stream += output_stream
        status = 'written'
    else:
        mseed_path = None
        status = 'no_output_traces'

    shot_rows.append({
        'virtual_shot_number': int(shot.virtual_shot_number),
        'virtual_shot_id': shot.virtual_shot_id,
        'source_cluster_id': shot.source_cluster_id,
        'source_x_m': float(shot.source_x_m),
        'component': COMPONENT,
        'n_planned_products': len(shot_products),
        'n_loaded_candidate_traces': n_loaded_candidate_traces,
        'n_output_traces': len(output_stream),
        'n_product_errors': len(product_errors),
        'product_errors_json': json.dumps(product_errors),
        'mseed_path': str(mseed_path) if mseed_path else None,
        'status': status,
    })
    if shot.virtual_shot_number % 20 == 0:
        print(f'Processed shot {shot.virtual_shot_number}/{len(shots)} at x={shot.source_x_m:.1f} m')

virtual_shot_manifest = pd.DataFrame(shot_rows)
virtual_trace_manifest = pd.DataFrame(trace_rows)
trace_contributions = pd.DataFrame(contribution_rows)
alignment_qc = pd.DataFrame(alignment_rows)
alignment_pair_qc = pd.DataFrame(alignment_pair_rows)
product_errors = pd.DataFrame(product_error_rows)

print('Written shot gathers:', int(virtual_shot_manifest.status.eq('written').sum()))
print('Output traces:', len(virtual_trace_manifest))

/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


Processed shot 20/159 at x=92.5 m
Processed shot 40/159 at x=108.0 m
Processed shot 60/159 at x=120.0 m
Processed shot 80/159 at x=132.0 m
Processed shot 100/159 at x=144.5 m
Processed shot 120/159 at x=163.0 m
Processed shot 140/159 at x=189.0 m
Written shot gathers: 159
Output traces: 10431


## 5. Write all-shot MiniSEED and catalogs

In [5]:
ALL_SHOTS_MSEED = OUT_ROOT / f'T1_virtual_all_shots_{COMPONENT}.mseed'

if WRITE_ALL_SHOTS_MSEED and len(all_shot_stream):
    all_shot_stream.traces.sort(key=lambda tr: (
        int(tr.stats.virtual_shot_number), float(tr.stats.receiver_x_m)
    ))
    all_shot_stream.write(str(ALL_SHOTS_MSEED), format='MSEED', encoding=MSEED_ENCODING)
    print('Wrote:', ALL_SHOTS_MSEED)

OUTPUTS = {
    'shots': OUT_ROOT / '99_virtual_T1_shot_manifest.csv',
    'traces': OUT_ROOT / '99_virtual_T1_trace_manifest.csv',
    'contributions': OUT_ROOT / '99_virtual_T1_trace_contributions.csv',
    'alignment_qc': OUT_ROOT / '99_virtual_T1_alignment_qc.csv',
    'alignment_pair_qc': OUT_ROOT / '99_virtual_T1_alignment_pair_qc.csv',
    'product_errors': OUT_ROOT / '99_virtual_T1_product_errors.csv',
    'summary': OUT_ROOT / '99_virtual_T1_build_summary.csv',
}

virtual_shot_manifest.to_csv(OUTPUTS['shots'], index=False)
virtual_trace_manifest.to_csv(OUTPUTS['traces'], index=False)
trace_contributions.to_csv(OUTPUTS['contributions'], index=False)
alignment_qc.to_csv(OUTPUTS['alignment_qc'], index=False)
alignment_pair_qc.to_csv(OUTPUTS['alignment_pair_qc'], index=False)
product_errors.to_csv(OUTPUTS['product_errors'], index=False)

planned_geode_products = int(products.product_kind.eq('geode_raw').sum())
geode_output_traces = int(virtual_trace_manifest.receiver_family.eq('geode').sum()) if len(virtual_trace_manifest) else 0
geode_product_errors = int(product_errors.product_kind.eq('geode_raw').sum()) if len(product_errors) else 0

if REQUIRE_GEODE_OUTPUT_IF_PLANNED and planned_geode_products > 0 and geode_output_traces == 0:
    display(product_errors.loc[product_errors.product_kind.eq('geode_raw')].head(50))
    raise RuntimeError(
        f'Notebook 98 planned {planned_geode_products} raw Geode products, but notebook 99 produced zero Geode output traces. See {OUTPUTS["product_errors"]}'
    )

summary = pd.DataFrame([
    ('virtual_shots_planned', len(shots)),
    ('virtual_shots_written', int(virtual_shot_manifest.status.eq('written').sum())),
    ('virtual_shots_empty', int(virtual_shot_manifest.status.ne('written').sum())),
    ('output_traces', len(virtual_trace_manifest)),
    ('T1_1m_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('T1_1m').sum()) if len(virtual_trace_manifest) else 0),
    ('T1_2m_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('T1_2m').sum()) if len(virtual_trace_manifest) else 0),
    ('nodal_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('nodal').sum()) if len(virtual_trace_manifest) else 0),
    ('T1_Streamer_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('T1_Streamer').sum()) if len(virtual_trace_manifest) else 0),
    ('planned_geode_products', planned_geode_products),
    ('geode_output_traces', geode_output_traces),
    ('geode_product_errors', geode_product_errors),
    ('alignment_products_accepted', int(alignment_qc.alignment_accepted.sum()) if len(alignment_qc) else 0),
    ('alignment_products_rejected', int((~alignment_qc.alignment_accepted).sum()) if len(alignment_qc) else 0),
    ('super_stacked_contributions', int(trace_contributions.super_stacked.fillna(False).sum()) if len(trace_contributions) else 0),
    ('output_start_s', OUTPUT_START_S),
    ('output_end_s', OUTPUT_END_S),
], columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print()
print('Written:')
for name, path in OUTPUTS.items():
    print(f'  {name:20s} {path}')

Wrote: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/T1_virtual_all_shots_Z.mseed


,metric,value
0,virtual_shots_planned,159.000
1,virtual_shots_written,159.000
2,virtual_shots_empty,0.000
3,output_traces,10431.000
4,T1_1m_primary_output_traces,2808.000
5,T1_2m_primary_output_traces,2592.000
6,nodal_primary_output_traces,4177.000
7,T1_Streamer_primary_output_traces,854.000
8,planned_geode_products,155.000
9,geode_output_traces,6254.000



Written:
  shots                /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_shot_manifest.csv
  traces               /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_trace_manifest.csv
  contributions        /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_trace_contributions.csv
  alignment_qc         /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_alignment_qc.csv
  alignment_pair_qc    /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_alignment_pair_qc.csv
  product_errors       /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_product_errors.csv
  summary              /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_build_summary.csv


## 6. Next step

Notebook 100 reads these ordered shot gathers and catalogs and exports:

- sparse all-shot SEG-Y, containing only observed traces;
- regularized all-shot SEG-Y, containing the full master receiver grid with
  missing combinations written as zero-valued traces marked dead;
- optional per-shot SEG-Y files.

The CSV manifests remain authoritative for receiver family and provenance.